# Markov Chains

---
In class today we will be implementing a Markov chain to process sentences

---
## Learning Objectives

1. Students will be able to explain the Markov Chain process
1. Implement a Markov Chain


Markov Chains represent a series of events following the Markov Property: future states are memory-less in that they depend only on the current state. This can be expanded to the idea of variable order Markov models where there is a variable-length memory (eg. 1st order Markov Model). Markov models consist of fully observable states. 

> A common example of this is in predicting the weather: We can clearly see the current weather and would like to predict tomorrow's weather. This is also applicable to biology with one case being CpG islands. 

Our goal today will be to implement a Markov model built from words. For our example text, we will use the classic example of Dr. Seuss because of the repetitive nature of the text.

---
## Train Markov model

For our initial implementation of the Markov Model, we will use the simple example of Dr. Seuss: "One fish two fish red fish blue fish."



In [28]:
def build_markov_model(markov_model:dict, new_text:str) -> dict:
    '''
    Function to build or add to a 1st order Markov model given a string of text
    We will store the markov model as a dictionary of dictionaries
    The key in the outer dictionary represents the current state
    and the inner dictionary represents the next state with their contents containing
    the transition probabilities.
    Note: This would be easier to read if we were to build a class representation
           of the model rather than a dictionary of dictionaries, but for simplicitiy
           our implementation will just use this structure.
    
    Args: 
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)
        new_text (str): a string to build or add to the moarkov_model

    Returns:
        markov_model (dict of dicts): an updated markov_model
        
    Pseudocode:
        Add artificial states for start and end
        For each word in text:
            Increment markov_model[word][next_word]
        
    '''
     # Split string of words into list of states
    states = new_text.split(' ')
    ad_states = ["*S*"] + states + ["*E*"]

    for index in range(len(ad_states) - 1):
        current_state = ad_states[index]
        next_state = ad_states[index + 1]
        # Check if current state in markov dict else add
        if current_state not in markov_model:
            markov_model[current_state] = {}

        # Check if next state in current state
        if next_state not in markov_model[current_state]:
            markov_model[current_state][next_state] = 0
        markov_model[current_state][next_state] += 1

    return markov_model

In [29]:
markov_model = dict()
text = "one fish two fish red fish blue fish"
markov_model = build_markov_model(markov_model, text)
print (markov_model)

{'*S*': {'one': 1}, 'one': {'fish': 1}, 'fish': {'two': 1, 'red': 1, 'blue': 1, '*E*': 1}, 'two': {'fish': 1}, 'red': {'fish': 1}, 'blue': {'fish': 1}}


###  Nth order Markov chain
In the above model, each event or word is output from only the previous state with no memory of any prior states. While this is useful in some cases, typical biological applications of Markov chains require higher-order models to accurately capture what we know about a system. For instance, in attempting to identify coding regions of a genome, we know that open reading frames (ORFs) contain codon triplets, and so a third or sixth order Markov chain would better describe these regions. Here you will implement a generalized form of our previous Markov Chain to allow for Nth order chains.


In [80]:
def build_markov_model(markov_model: dict, text: str, order: int) -> dict:
    '''
    Function to build or add to a Nth order Markov model given a string of text

    Args: 
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)
            or None if a new model is being built
        new_text (str): a string to build or add to the moarkov_model
        order (int): the number of previous states to consider for the model
        
    Returns:
        markov_model (dict of dicts): an updated/new markov_model
    '''
# Split string of words into list of states
    words = text.split(' ')
    states = ["*S*"] * order + words + ["*E*"]

    for index in range(len(states) - order):
        current_state = tuple(states[index: index + order])
        next_state = states[index + order]

        if current_state not in markov_model:
            markov_model[current_state] = {}

        if next_state not in markov_model[current_state]:
            markov_model[current_state][next_state] = 0
        markov_model[current_state][next_state] += 1

    return markov_model

In [41]:
markov_model = dict()
text = "one fish two fish red fish blue fish"
markov_model = build_markov_model(markov_model, text, order=2)
markov_model

{('*S*', '*S*'): {'one': 1},
 ('*S*', 'one'): {'fish': 1},
 ('one', 'fish'): {'two': 1},
 ('fish', 'two'): {'fish': 1},
 ('two', 'fish'): {'red': 1},
 ('fish', 'red'): {'fish': 1},
 ('red', 'fish'): {'blue': 1},
 ('fish', 'blue'): {'fish': 1},
 ('blue', 'fish'): {'*E*': 1}}

## Generate text from Markov Model

Markov models are "generative models". That is, the probability states in the model can be used to generate output following the conditional probabilities in the model.

We will now generate a sequence of text from the Markov model. For this section, I recommend using np.random.choice, which allows for you to provide a probability distribution for drawing the next edge in the chain.

In [81]:
import numpy as np

def get_next_word(current_word, markov_model, seed=42):
    '''
    Function to randomly move a valid next state given a markov model
    and a current state (word)
    
    Args: 
        current_word (tuple): a word that exists in our model
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)

    Returns:
        next_word (str): a randomly selected next word based on transition probabilies
        
    Pseudocode:
        Calculate transition probilities for all next states from a given state (counts/sum)
        Randomly draw from these to generate the next state
        
    '''

    if current_word not in markov_model:
        return "*E*"

    next_counts = markov_model[current_word]
    next_words = list(next_counts.keys())
    counts = np.array(list(next_counts.values()), dtype=float)

    total = counts.sum()
    probabilities = counts / total

    if total <= 0:
        return "*E*"

    if not np.isclose(probabilities.sum(),1.0):
        raise ValueError("Probabilities must sum to 1.")

    return np.random.choice(next_words, p=probabilities)

def generate_random_text(markov_model, seed=42):
    '''
    Function to generate text given a markov model
    
    Args: 
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)

    Returns:
        sentence (str): a randomly generated sequence given the model
        
    Pseudocode:
        Initialize sentence at start state
        Until End State:
            append get_next_word(current_word, markov_model)
        Return sentence
        
    '''
    np.random.seed(seed)

    start = "*S*"
    end = "*E*"

    # Get nth order from markov_model
    for state in markov_model:
        order = int(len(state))
        break

    current_state = tuple([start] * order)

    words = []

    # Loop until transition to end state
    while True:
        next_word = get_next_word(current_state, markov_model)

        # Break loop if next_word is end state
        if next_word == end:
            break

        # append next word
        words.append(next_word)

        current_state = current_state[1:] + (next_word,)

    return " ".join(words)

---

In [50]:
markov_model = dict()
text = "one fish two fish red fish blue fish"
markov_model = build_markov_model(markov_model, text, order=2)
generate_random_text(markov_model, seed=42)

'one fish two fish red fish blue fish'

## All the Fish
Up till now, you have only been working with a line or two of the Dr. Seuss' _One Fish, Two Fish_. Now, I want you to build a model using the whole book and try different orders of Markov models.

In [83]:
# Now just add some more training data to the markov model. You can find it under data/one_fish_two_fish.txt

# Generate one line
markov_model = dict()
# Read in the whole book
file_path = "data/one_fish_two_fish.txt"
order = 2

with open(file_path, "r", encoding="utf-8") as infile:
    for line in infile:
        line = line.strip()
        if not line:
            continue
        markov_model = build_markov_model(markov_model, line, order)

print(generate_random_text(markov_model, seed=7))


Some have two feet and some are blue.


In [84]:
# Now just add some more training data to the markov model. You can find it under data/one_fish_two_fish.txt
#Generate paragraph
markov_model = dict()
# Read in the whole book
file_path = "data/one_fish_two_fish.txt"
order = 2

book = ""

with open(file_path, "r", encoding="utf-8") as infile:
    for line in infile:
        line = line.strip()
        if line == "":
            if book != "":
                markov_model = build_markov_model(markov_model, book, order)
                book = ""
        else:
            book = book + " " + line + "\n"

if book:
    markov_model = build_markov_model(markov_model, book, order)
print(generate_random_text(markov_model, seed=7))


 One fish, Two fish, Red fish, Blue fish,
 Black fish, Blue fish,
 Black fish, Blue fish,
 Black fish, Blue fish, Old fish, New fish.
 This one is quiet as a mouse.
 Oh! What a house!
 Oh dear, oh dear! I cannot hear your call.
 I cannot hear your call at all.
 This is no good. This is not good, and I know why.
 A mouse has cut the wire, goodbye!
 From near to far, from here to there,
 Funny things are everywhere.



---
## Shakespeare

Now, let's play around with some Shakespeare.

In [85]:
# An example of a more complex text that we can use to generate more complex output
sonet_markov_model = dict()
file = open("data/sonnets.txt", "r")
sonet = ""
for line in file:
    line = line.strip()
    if line == "":
        # Empty line so build model
        sonet_markov_model = build_markov_model(sonet_markov_model, sonet, order=2)
        sonet = ""
    else:
        sonet = sonet + ' ' + line + "\n"
 
print (generate_random_text(sonet_markov_model,seed=7))

 Say that thou forget'st so long,
 To speak of that which it fears to hopes, and hopes to fears,
 Still losing when I forgot
 Am of my harmful deeds,
 That did my ripe thoughts in my mind,
 My grief lies onward, and my will one.
 In things of great receipt with ease we prove
 Among a number one is reckon'd none:
 Then in the main of light,
 Crawls to maturity, wherewith being crown'd,
 Crooked eclipses 'gainst his glory fight,
 And Time that gave doth now his gift confound.
 Time doth transfix the flourish set on youth
 And delves the parallels in beauty's brow,
 Feeds on the ashes of his living hue?
 Why should my heart hath 'scap'd this sorrow,
 Come in the breath that from my face she turns my foes,
 That they behold, and see not what they despise,
 Who, in despite of space I would be brought,
 From limits far remote, where thou dost hide,
 By self-example mayst thou be good, slander doth but approve
 Thy worth the greater being woo'd of time;
 For canker vice the sweetest buds doth